In [ ]:
# Audio Processing imports
from IPython.display import Audio, display
import librosa
import librosa.display
import pandas as pd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import sys
from pathlib import Path


# Dataset loading and inspection

In [ ]:
project_dir = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src' / 'datasets.py').exists())
sys.path.insert(0, str(project_dir / 'src'))

from datasets import dataset_path

la_dir = dataset_path('asvspoof19_la')
print('ASVspoof 2019 LA available:', la_dir.exists())

# Use protocol entries so every selected clip has a label.
protocol_dir = la_dir / 'ASVspoof2019_LA_cm_protocols'
splits = {}
for split, suffix in [('train', 'trn'), ('dev', 'trl'), ('eval', 'trl')]:
    protocol = protocol_dir / f'ASVspoof2019.LA.cm.{split}.{suffix}.txt'
    frame = pd.read_csv(protocol, sep=r'\s+', names=['speaker', 'sample_id', 'unused', 'attack', 'key'])
    audio_dir = la_dir / f'ASVspoof2019_LA_{split}' / 'flac'
    frame['path'] = frame['sample_id'].map(lambda name: audio_dir / f'{name}.flac')
    frame['label'] = frame['key'].map({'bonafide': 0, 'spoof': 1})
    assert frame['label'].notna().all()
    assert frame['sample_id'].is_unique
    assert frame['path'].map(Path.exists).all()
    splits[split] = frame
    extra = len(set(audio_dir.glob('*.flac')) - set(frame['path']))
    print(split, frame['key'].value_counts().to_dict(), 'extra files:', extra)

train = splits['train']
examples = train.groupby('key').sample(n=5, random_state=42).copy()
info = examples['path'].map(sf.info)
examples['sample_rate'] = info.map(lambda item: item.samplerate)
examples['channels'] = info.map(lambda item: item.channels)
examples['duration'] = info.map(lambda item: item.duration)
display(examples[['sample_id', 'key', 'sample_rate', 'channels', 'duration']])

# Audio Processing

In [ ]:
# Starting settings: 16 kHz mono, edge trim, peak normalization, 4 seconds.
sample_rate = 16000
window_seconds = 4

def process_audio(path, trim=True):
    audio, _ = librosa.load(path, sr=sample_rate, mono=True)
    if trim:
        audio, _ = librosa.effects.trim(audio, top_db=30)
    audio = librosa.util.normalize(audio)
    return librosa.util.fix_length(audio, size=sample_rate * window_seconds)

# Inspect one clip per class before extracting a batch.
from IPython.display import Audio, display

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for row, key in enumerate(['bonafide', 'spoof']):
    path = examples.loc[examples['key'] == key, 'path'].iloc[0]
    original, _ = librosa.load(path, sr=sample_rate, mono=True)
    processed = process_audio(path)
    for ax, audio, title in zip(axes[row], [original, processed], ['Original', 'Processed']):
        ax.plot(np.arange(len(audio)) / sample_rate, audio)
        ax.set(title=f'{key}: {title}', xlabel='Time (seconds)', ylabel='Amplitude')
    print(key, path.stem)
    display(Audio(original, rate=sample_rate))
    display(Audio(processed, rate=sample_rate))
plt.tight_layout()
plt.show()

# MFCC baseline

20 coefficients with 25 ms windows and 10 ms hops. Mean and standard deviation over time produce 40 features per clip. Padding influences these statistics; keep settings fixed and compare trimmed/untrimmed audio later. This section checks feature extraction only, before training a classifier.

In [ ]:
def extract_mfcc(path, trim=True):
    audio = process_audio(path, trim=trim)
    return librosa.feature.mfcc(
        y=audio, sr=sample_rate, n_mfcc=20,
        n_fft=400, hop_length=160, n_mels=40,
    )

mfcc = extract_mfcc(examples.iloc[0]['path'])
fig, ax = plt.subplots(figsize=(10, 4))
image = librosa.display.specshow(mfcc, sr=sample_rate, hop_length=160, x_axis='time', ax=ax)
ax.set(title='MFCCs', ylabel='MFCC coefficient')
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

# Balanced smoke test, not the final training distribution.
subset = train.groupby('key').sample(n=50, random_state=42).copy()
matrices = [extract_mfcc(path) for path in subset['path']]
X = np.stack([np.concatenate([m.mean(axis=1), m.std(axis=1)]) for m in matrices])
labels = subset['label'].to_numpy()
sample_ids = subset['sample_id'].to_numpy()
assert X.shape == (100, 40)
assert np.isfinite(X).all()
print('MFCC matrix:', mfcc.shape)
print('Features:', X.shape, 'labels:', labels.shape)
print('Class counts:', subset['key'].value_counts().to_dict())

# Experiment settings and MFCC classifier

Default: balanced smoke subsets, not final research results. Set `FULL_RUN = True` for all protocol-listed clips. Validation selects thresholds; test and external labels never tune models. Outputs and caches stay under ignored `outputs/`. The earlier 100-clip cell is only an extraction check.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, confusion_matrix, ConfusionMatrixDisplay

FULL_RUN = False
seed = 42
run_name = 'full' if FULL_RUN else 'smoke'
output_dir = project_dir / 'outputs' / 'audio' / run_name
output_dir.mkdir(parents=True, exist_ok=True)
frames = {name: frame.copy() if FULL_RUN else frame.groupby('key').sample(n=20, random_state=seed).copy()
          for name, frame in splits.items()}
frames = {('validation' if name == 'dev' else 'test' if name == 'eval' else name): frame
          for name, frame in frames.items()}

def summarize_mfcc(audio):
    matrix = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=20, n_fft=400, hop_length=160, n_mels=40)
    return np.concatenate([matrix.mean(axis=1), matrix.std(axis=1)])

def mfcc_features(frame, trim=True, condition='clean'):
    rng = np.random.default_rng(seed)
    values = []
    for path in frame['path']:
        audio = process_audio(path, trim=trim)
        audio = degrade_audio(audio, condition, rng)
        values.append(summarize_mfcc(audio))
    return np.stack(values)

def degrade_audio(audio, condition, rng):
    if condition == 'noise':
        # Add white noise at 10 dB SNR, measured over the fixed window.
        noise = rng.standard_normal(len(audio))
        noise *= np.sqrt(np.mean(audio ** 2) / (10 * np.mean(noise ** 2)))
        return (audio + noise).astype(np.float32)
    if condition == 'mulaw':
        # Simulated 8-bit mu-law quantization; not a full telephone codec.
        audio = np.clip(audio, -1, 1)
        encoded = np.sign(audio) * np.log1p(255 * np.abs(audio)) / np.log1p(255)
        encoded = np.round((encoded + 1) * 127.5) / 127.5 - 1
        return (np.sign(encoded) * np.expm1(np.abs(encoded) * np.log1p(255)) / 255).astype(np.float32)
    assert condition == 'clean'
    return audio

def new_classifier():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=seed))

mfcc_X = {name: mfcc_features(frame) for name, frame in frames.items()}
targets = {name: frame['label'].to_numpy() for name, frame in frames.items()}
mfcc_model = new_classifier()
mfcc_model.fit(mfcc_X['train'], targets['train'])
print(run_name, {name: values.shape for name, values in mfcc_X.items()})

# Pretrained encoder model

In [ ]:
import torch
from transformers import AutoFeatureExtractor, AutoModel

encoder_name = 'facebook/wav2vec2-base'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = AutoFeatureExtractor.from_pretrained(encoder_name)
encoder = AutoModel.from_pretrained(encoder_name).to(device).eval()
encoder.requires_grad_(False)
assert processor.sampling_rate == sample_rate

def encoder_features(frame, cache_name=None, condition='clean'):
    # One clip at a time limits GPU memory; cache fixed-window embeddings.
    cache = output_dir / f'wav2vec2-base_16k_4s_trim30_{cache_name}.npz' if cache_name else None
    ids = frame['sample_id'].to_numpy(dtype=str)
    if cache and cache.exists():
        with np.load(cache) as saved:
            if np.array_equal(saved['sample_ids'], ids):
                return saved['features']
    rng = np.random.default_rng(seed)
    values = []
    for path in frame['path']:
        audio = degrade_audio(process_audio(path), condition, rng)
        inputs = processor(audio, sampling_rate=sample_rate, return_tensors='pt')
        inputs = {key: value.to(device) for key, value in inputs.items()}
        with torch.inference_mode():
            hidden = encoder(**inputs).last_hidden_state
        values.append(hidden.mean(dim=1).squeeze(0).cpu().numpy())
    features = np.stack(values)
    if cache:
        np.savez_compressed(cache, sample_ids=ids, features=features)
    return features

encoder_X = {name: encoder_features(frame, cache_name=name) for name, frame in frames.items()}
encoder_model = new_classifier()
encoder_model.fit(encoder_X['train'], targets['train'])
print('Device:', device, 'Embedding shape:', encoder_X['train'].shape)

# In-Domain evaluation

In [ ]:
# Select the threshold maximizing validation F1; never select on test labels.
def select_threshold(labels, scores):
    from sklearn.metrics import precision_recall_curve
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(thresholds[np.argmax(f1)])

def evaluate_scores(labels, scores, threshold):
    fpr, tpr, _ = roc_curve(labels, scores)
    fnr = 1 - tpr
    # Discrete ROC approximation to EER, not an operational threshold.
    index = np.argmin(np.abs(fpr - fnr))
    return {'roc_auc': roc_auc_score(labels, scores),
            'eer': float((fpr[index] + fnr[index]) / 2),
            'f1': f1_score(labels, scores >= threshold, zero_division=0),
            'threshold': threshold, 'n': len(labels)}

models = {'mfcc': mfcc_model, 'wav2vec2': encoder_model}
feature_sets = {'mfcc': mfcc_X, 'wav2vec2': encoder_X}
scores = {}
thresholds = {}
results = []
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, model) in zip(axes, models.items()):
    scores[name] = {split: model.predict_proba(values)[:, 1] for split, values in feature_sets[name].items()}
    thresholds[name] = select_threshold(targets['validation'], scores[name]['validation'])
    for split in ['validation', 'test']:
        results.append({'model': name, 'dataset': split, 'condition': 'clean',
                        **evaluate_scores(targets[split], scores[name][split], thresholds[name])})
    ConfusionMatrixDisplay(confusion_matrix(targets['test'], scores[name]['test'] >= thresholds[name]),
                           display_labels=['bonafide', 'spoof']).plot(ax=ax, colorbar=False)
    ax.set_title(f'{name}: test ({run_name})')
plt.tight_layout()
plt.show()
display(pd.DataFrame(results))

for name in models:
    fpr, tpr, _ = roc_curve(targets['test'], scores[name]['test'])
    plt.plot(fpr, tpr, label=name)
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.legend()
plt.title(f'Test ROC ({run_name})')
plt.show()

# Robustness and ablations
Compare clean-trained models on degraded test audio. Separately retrain MFCC baselines without trimming and with training-only noise/mu-law augmentation. Each retrained model selects its own threshold on matching clean validation data. Encoder trimming/augmentation retraining is not included. min t-DCF is not computed: it requires a matching ASV-score/cost protocol; AUC/EER alone are not substitutes.

In [ ]:
degraded_mfcc = {}
for condition in ['noise', 'mulaw']:
    degraded_mfcc[condition] = mfcc_features(frames['test'], condition=condition)
    degraded_encoder = encoder_features(frames['test'], condition=condition)
    for name, features in [('mfcc', degraded_mfcc[condition]), ('wav2vec2', degraded_encoder)]:
        prediction = models[name].predict_proba(features)[:, 1]
        results.append({'model': name, 'dataset': 'test', 'condition': condition,
                        **evaluate_scores(targets['test'], prediction, thresholds[name])})

untrimmed_X = {name: mfcc_features(frame, trim=False) for name, frame in frames.items()}
untrimmed_model = new_classifier().fit(untrimmed_X['train'], targets['train'])
untrimmed_threshold = select_threshold(targets['validation'], untrimmed_model.predict_proba(untrimmed_X['validation'])[:, 1])
prediction = untrimmed_model.predict_proba(untrimmed_X['test'])[:, 1]
results.append({'model': 'mfcc_untrimmed', 'dataset': 'test', 'condition': 'clean',
                **evaluate_scores(targets['test'], prediction, untrimmed_threshold)})

augmented_X = np.concatenate([mfcc_X['train']] + [mfcc_features(frames['train'], condition=c) for c in ['noise', 'mulaw']])
augmented_model = new_classifier().fit(augmented_X, np.tile(targets['train'], 3))
augmented_threshold = select_threshold(targets['validation'], augmented_model.predict_proba(mfcc_X['validation'])[:, 1])
for condition, values in {'clean': mfcc_X['test'], **degraded_mfcc}.items():
    prediction = augmented_model.predict_proba(values)[:, 1]
    results.append({'model': 'mfcc_augmented', 'dataset': 'test', 'condition': condition,
                    **evaluate_scores(targets['test'], prediction, augmented_threshold)})
display(pd.DataFrame(results))

# External evaluation: In-the-Wild
Set `IN_THE_WILD_DIR` in `.env` to the directory containing `meta.csv` and the WAV files. Expected metadata columns: `file`, `speaker`, `label`, with labels `bona-fide` and `spoof`. Confirm this layout against your downloaded release. Missing data is reported as skipped, never as evaluated. No retraining or threshold tuning on external data.

In [ ]:
external_dir = dataset_path('in_the_wild')
external = None
external_scores = {}
if external_dir is None or not (external_dir / 'meta.csv').exists():
    print('External evaluation skipped: configure In-the-Wild data and meta.csv.')
else:
    external = pd.read_csv(external_dir / 'meta.csv')
    external['key'] = external['label'].astype(str).str.strip().map({'bona-fide': 'bonafide', 'bonafide': 'bonafide', 'spoof': 'spoof'})
    assert external['key'].notna().all(), 'Inspect unexpected external labels before evaluation'
    external['label'] = external['key'].map({'bonafide': 0, 'spoof': 1})
    external['sample_id'] = 'itw_' + external['file'].astype(str)
    external['path'] = external['file'].map(lambda name: external_dir / str(name))
    assert external['sample_id'].is_unique
    assert external['path'].map(Path.exists).all()
    if not FULL_RUN:
        external = external.groupby('key').sample(n=20, random_state=seed).copy()
    external_X = {'mfcc': mfcc_features(external), 'wav2vec2': encoder_features(external, cache_name='external')}
    for name, model in models.items():
        external_scores[name] = model.predict_proba(external_X[name])[:, 1]
        results.append({'model': name, 'dataset': 'in_the_wild', 'condition': 'clean',
                        **evaluate_scores(external['label'], external_scores[name], thresholds[name])})
    display(pd.DataFrame(results).query("dataset == 'in_the_wild'"))

# Score export and conclusions
Higher score means higher spoof probability (`1 = spoof`, `0 = bonafide`). Training exports use out-of-fold predictions with scaling fitted inside each fold; validation/test use the final training-fitted model. These are sigmoid classifier probabilities, not independently calibrated risks. Fusion must align IDs using its pairing manifest; audio scores alone do not establish scam intent. Smoke exports are for integration checks only.

In [ ]:
import json
import joblib

folds = StratifiedKFold(n_splits=5 if FULL_RUN else 3, shuffle=True, random_state=seed)
for name, model in models.items():
    oof = cross_val_predict(new_classifier(), feature_sets[name]['train'], targets['train'],
                             cv=folds, method='predict_proba')[:, 1]
    tables = []
    for split, frame in frames.items():
        table = frame[['sample_id', 'label']].copy()
        table['split'] = split
        table['score'] = oof if split == 'train' else scores[name][split]
        tables.append(table[['sample_id', 'split', 'label', 'score']])
    pd.concat(tables, ignore_index=True).to_csv(output_dir / f'{name}_scores.csv', index=False)
    joblib.dump(model, output_dir / f'{name}_classifier.joblib')
    if external is not None:
        table = external[['sample_id', 'label']].copy()
        table['split'] = 'external'
        table['score'] = external_scores[name]
        table[['sample_id', 'split', 'label', 'score']].to_csv(output_dir / f'{name}_external_scores.csv', index=False)

metrics = pd.DataFrame(results)
metrics.to_csv(output_dir / 'metrics.csv', index=False)
settings = {'mode': run_name, 'seed': seed, 'sample_rate': sample_rate, 'window_seconds': window_seconds,
            'trim_top_db': 30, 'crop': 'start', 'padding': 'right_zero', 'normalization': 'peak',
            'n_mfcc': 20, 'n_fft': 400, 'hop_length': 160, 'n_mels': 40,
            'encoder': encoder_name, 'encoder_revision': getattr(encoder.config, '_commit_hash', None),
            'pooling': 'time_mean', 'threshold_selection': 'maximum_validation_f1',
            'thresholds': thresholds, 'train_scores': 'stratified_out_of_fold',
            'external_evaluated': external is not None}
(output_dir / 'settings.json').write_text(json.dumps(settings, indent=2))
for name in models:
    errors = frames['test'][['sample_id', 'key', 'attack']].copy()
    errors['score'] = scores[name]['test']
    errors['prediction'] = (errors['score'] >= thresholds[name]).astype(int)
    errors = errors.loc[errors['prediction'].to_numpy() != targets['test']]
    errors.to_csv(output_dir / f'{name}_test_errors.csv', index=False)
    print(name, 'test errors:', len(errors))
print('Saved:', output_dir)
display(metrics)

## Interpretation checklist after your full run

- Compare MFCC versus wav2vec 2.0 on test AUC, EER, and F1.
- Explain the trimmed/untrimmed difference and padding limitations.
- Compare clean/degraded performance and whether training augmentation helps.
- Report external generalization separately, or explicitly report missing data.
- Listen to representative false positives and false negatives from the error CSVs.
- Record that ASVspoof 2021 LA is optional and not implemented here; min t-DCF remains uncomputed.
- For full runs, set `FULL_RUN = True`, restart the kernel, and run all cells. Delete embedding caches after changing preprocessing or encoder settings. Smoke results do not support final project claims.